# Notebook 4: Attention Mechanism — Deep Dive

**Goal**: Build and visualize the attention mechanism step by step.
Watch the causal mask work. See how multiple heads attend to different relationships.

After this notebook you will:
- Be able to compute Q, K, V projections by hand
- Understand exactly what softmax does to the scores
- See how causal masking prevents attending to the future

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

from 04_attention.scaled_dot_product import scaled_dot_product_attention, generate_causal_mask
from 04_attention.multi_head_attention import MultiHeadAttention

torch.manual_seed(42)
print('Ready')

## 1. Step-by-step attention on a toy example

Let's trace through the formula for the sentence: **"marathon pace was fast"**

In [ ]:
# Toy embeddings: 4 tokens, d_model=4
torch.manual_seed(0)
B, T, d_model = 1, 4, 4
d_k = d_model   # single head for clarity

tokens = ['marathon', 'pace', 'was', 'fast']
x = torch.randn(B, T, d_model)   # pretend these are real embeddings

# Projection matrices (random for demo; trained in real model)
W_q = torch.randn(d_model, d_k)
W_k = torch.randn(d_model, d_k)
W_v = torch.randn(d_model, d_k)

# Project
Q = x @ W_q   # [1, 4, 4]
K = x @ W_k
V = x @ W_v
print(f'Q shape: {Q.shape} | K shape: {K.shape} | V shape: {V.shape}')

In [ ]:
# Step 1: Dot product (raw scores)
scores_raw = (Q @ K.transpose(-2, -1))
print('Raw scores (Q·Kᵀ):')
print(scores_raw[0].detach().numpy().round(2))

In [ ]:
# Step 2: Scale
scores_scaled = scores_raw / math.sqrt(d_k)
print(f'After dividing by √{d_k}={math.sqrt(d_k):.2f}:')
print(scores_scaled[0].detach().numpy().round(2))

In [ ]:
# Step 3: Apply causal mask
mask = generate_causal_mask(T)
print('Causal mask (True = can attend):')
print(mask[0, 0].numpy())

scores_masked = scores_scaled.masked_fill(mask == 0, float('-inf'))
print('\nAfter masking (-inf = blocked):')
print(scores_masked[0].detach().numpy().round(2))

In [ ]:
# Step 4: Softmax -> attention weights
weights = F.softmax(scores_masked, dim=-1)
print('Attention weights (each row sums to 1.0):')
for i, (tok, row) in enumerate(zip(tokens, weights[0])):
    row_np = row.detach().numpy().round(3)
    print(f'  {tok:<10} attends to: {dict(zip(tokens[:i+1], row_np[:i+1]))}')

In [ ]:
# Step 5: Weighted sum of values
output = weights @ V
print(f'Output shape: {output.shape}  (same as input: [B, T, d_k])')
print('Each token now has a new representation = weighted blend of all Values it attended to')

## 2. Visualize the attention weight matrix

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Without mask
weights_no_mask = F.softmax(scores_scaled, dim=-1)[0].detach().numpy()
ax1.imshow(weights_no_mask, cmap='Blues', vmin=0, vmax=1)
ax1.set_xticks(range(T)); ax1.set_yticks(range(T))
ax1.set_xticklabels(tokens, rotation=45)
ax1.set_yticklabels(tokens)
ax1.set_title('Without causal mask\n(can see future — wrong for GPT)')

# With mask
weights_masked = weights[0].detach().numpy()
ax2.imshow(weights_masked, cmap='Blues', vmin=0, vmax=1)
ax2.set_xticks(range(T)); ax2.set_yticks(range(T))
ax2.set_xticklabels(tokens, rotation=45)
ax2.set_yticklabels(tokens)
ax2.set_title('With causal mask\n(can only see past — correct for GPT)')

plt.tight_layout()
plt.show()

## 3. Multi-head attention — parallel specialization

In [ ]:
# Use our actual MultiHeadAttention module
d_model = 32
num_heads = 4
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=0.0)

T = 8
x = torch.randn(1, T, d_model)
_, attn_weights = mha(x, return_weights=True)
print(f'Attention weights shape: {attn_weights.shape}  [B, num_heads, T, T]')

In [ ]:
# Plot all heads side by side
fig, axes = plt.subplots(1, num_heads, figsize=(4*num_heads, 4))

for h in range(num_heads):
    w = attn_weights[0, h].detach().numpy()
    axes[h].imshow(w, cmap='Blues', aspect='auto', vmin=0, vmax=w.max())
    axes[h].set_title(f'Head {h+1}')
    axes[h].set_xlabel('Key position')
    if h == 0:
        axes[h].set_ylabel('Query position')

plt.suptitle('Multi-head attention weights — each head learns different patterns', y=1.02)
plt.tight_layout()
plt.show()
print('Notice each head has a different attention pattern — they specialize!')

## 4. Verify mathematical properties

In [ ]:
# Property 1: weights sum to 1.0 for each query position
row_sums = attn_weights.sum(dim=-1)
print(f'Row sums (should all be 1.0): min={row_sums.min():.6f}, max={row_sums.max():.6f}')

# Property 2: causal mask — upper triangle must be zero
for h in range(num_heads):
    upper = torch.triu(attn_weights[0, h], diagonal=1)
    assert upper.abs().max() < 1e-6, f'Head {h} violated causal mask!'
print('Causal mask verified: no future positions attended to ✓')

# Property 3: output shape equals input shape
out, _ = mha(x)
print(f'Input shape:  {x.shape}')
print(f'Output shape: {out.shape}  (identical — this is what allows stacking)')

## Exercise

1. Remove the `/ math.sqrt(d_k)` scaling in Step 2. What happens to the attention weights?
   Are they more or less uniform? Why?
2. Set `num_heads=1`. Plot the attention. Then set `num_heads=8`. What changes?
3. What is the memory cost of the attention weight matrix [B, h, T, T] for
   T=256, h=4, B=32 (our training config)? What about T=4096 (GPT-4 scale)?
4. The causal mask is upper-triangular False. What if you made it fully True
   (bidirectional)? When would you want that? (Hint: think BERT vs GPT.)